In [1]:
import cv2
import numpy as np
import torch
import os
import sys
import time
from datetime import datetime
import matplotlib.pyplot as plt
import math
import gc # For garbage collection

# --- Add project root to Python path ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project Root: {project_root}")

MODEL_PATH = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine"
MODEL_INPUT_SIZE = 640
CONF_THRESHOLD = 0.05
IOU_THRESHOLD = 0.6
MODEL_INPUT_SIZE = 640
TARGET_FPS = 20
PYVIDEOREADER_DECODE_SEGMENT_SIZE = 500

from models.model_loader import load_model, get_device
from core.video_processor import VideoProcessor

from video_reader import PyVideoReader

# --- Helper Functions ---
def display_image_jupyter(image_bgr, title="Image"):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

print("Setup Complete. Necessary modules imported.")

Project Root: c:\Users\swest\Desktop\traffic-capturing\traffic-reporting
--- Configuration Loaded (Tuned for Exit Accuracy) ---
Start Date/Time: 250906 / 111618252
Using Model: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine (Conf: 0.05, IoU: 0.6)
Model Input Size: 640x640
Target FPS: 20, Batch Size: 64, Threads: 48
Mixed Precision: True, CUDA Streams: 32
Detailed Perf Metrics: True, Debug Mode: False
--------------------------------------------------------------------


Setup Complete. Necessary modules imported.


In [2]:
# --- Configuration for this Benchmark Run ---

VIDEO_PATH = r"C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4"
OUTPUT_DIR_BASE = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs"

# --- Configuration for this Benchmark Run ---

FRAME_INTERVAL_CONFIG = 25
CONF_THRESHOLD_OVERRIDE = None
MAX_FRAMES_TO_PROCESS = 416
DISPLAY_INTERACTIVE_FRAMES = None
SAVE_ANNOTATED_FRAMES = True

timestamp_run = datetime.now().strftime("%Y%m%d_%H%M")
video_name_stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem}_{timestamp_run}")

In [3]:
print("Initializing VideoProcessor instance for preprocessing...")
vp = VideoProcessor('hardcoded')

print("\nLoading model")
device = get_device() # Use global get_device for this script
effective_conf_thresh = CONF_THRESHOLD_OVERRIDE if CONF_THRESHOLD_OVERRIDE is not None else CONF_THRESHOLD
model = load_model(MODEL_PATH, device, effective_conf_thresh, IOU_THRESHOLD) # load_model takes device
vp.device = device # Ensure VideoProcessor instance uses the same device as the loaded model

print(f"Model loaded on device: {device}")

class_names_map = vp.class_id_to_name_map
# class_names_map = {}
default_class_label = 'unknown'

if not class_names_map:
    print(f"CRITICAL from VideoProcessor: Class name map is empty. Using generic names '{default_class_label}_ID'.")
    # Fallback if vp didn't load them for some reason (should not happen if YAML is correct)
    if hasattr(model, 'names') and model.names:
        class_names_map = model.names
        print(f"Fell back to model.names: {class_names_map}")
    else:
        class_names_map = {i: f"{default_class_label}_{i}" for i in range(80)} # Assume up to 80
        print(f"Fell back to very generic names. First few: { {k:v for i,(k,v) in enumerate(class_names_map.items()) if i<3} }")

print(f"Using class name map (from VideoProcessor instance): {class_names_map if class_names_map else 'EMPTY - CHECK VP INIT'}")

2025-09-06 11:16:20.102 | WARNING  | core.video_processor:__init__:69 - DATA_YAML_PATH ('') not found. Class names will be default.
2025-09-06 11:16:20.103 | DEBUG    | models.model_loader:load_model:11 - Loading model from: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine onto device: cuda
2025-09-06 11:16:20.103 | DEBUG    | models.model_loader:load_model:47 - Detected .engine file, using TensorRT runtime via YOLO wrapper.


Initializing VideoProcessor instance for preprocessing...
Using GPU: NVIDIA GeForce RTX 3090

Loading model
Using GPU: NVIDIA GeForce RTX 3090


2025-09-06 11:16:20.440 | DEBUG    | models.model_loader:load_model:50 - TensorRT engine loaded successfully from: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine


Model loaded on device: cuda
CRITICAL from VideoProcessor: Class name map is empty. Using generic names 'unknown_ID'.
Loading C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine for TensorRT inference...
Fell back to model.names: {0: 'Bus', 1: 'Cyclist', 2: 'Light Vehicle', 3: 'Long Truck', 4: 'Medium Truck', 5: 'Minibus Taxi', 6: 'Motorcycle', 7: 'Pedestrian', 8: 'Short Truck'}
Using class name map (from VideoProcessor instance): {0: 'Bus', 1: 'Cyclist', 2: 'Light Vehicle', 3: 'Long Truck', 4: 'Medium Truck', 5: 'Minibus Taxi', 6: 'Motorcycle', 7: 'Pedestrian', 8: 'Short Truck'}


------------------------------------------------

-----------------------------------

Using patched_yolo_infer

In [ ]:
# Import the patched_yolo_infer library
from patched_yolo_infer import MakeCropsDetectThem, CombineDetections
from ultralytics import YOLO

MAX_FRAMES_TO_PROCESS = 500
# Crop-based inference parameters (equivalent to SAHI)
CROP_HEIGHT = 640  
CROP_WIDTH = 640   
OVERLAP_HEIGHT_PERCENTAGE = 15  
OVERLAP_WIDTH_PERCENTAGE = 15   
POSTPROCESS_CONF_THRESHOLD = 0.15
BASE_MODEL_CONF_FOR_CROPS = 0.08
POSTPROCESS_IOU_THRESHOLD = 0.8
NMS_THRESHOLD = 0.8
timestamp_run = datetime.now().strftime("%Y%m%d_%H%M")

print(f"  Crop Params: Crop HxW={CROP_HEIGHT}x{CROP_WIDTH}, Overlap={OVERLAP_HEIGHT_PERCENTAGE}%, Post-Conf={POSTPROCESS_CONF_THRESHOLD}")

# Base Model Confidence for individual crops
if CONF_THRESHOLD_OVERRIDE is None or CONF_THRESHOLD_OVERRIDE > BASE_MODEL_CONF_FOR_CROPS:
    CONF_THRESHOLD_OVERRIDE = BASE_MODEL_CONF_FOR_CROPS
    print(f"  Override: CONF_THRESHOLD_OVERRIDE (for base model on crops) set to {CONF_THRESHOLD_OVERRIDE}")

model = load_model(MODEL_PATH, device, None, None)

# Advanced crop inference settings
USE_BATCH_INFERENCE = True  
USE_MEMORY_OPTIMIZE = False  
ENABLE_PROGRESS_TRACKING = False  
CLASS_AGNOSTIC_NMS = False  
INTELLIGENT_SORTER = True  
SORTER_BINS = 7  
MATCH_METRIC = 'IOS'

# Enhanced output directory structure
crop_str_override = "_crops"

if os.path.exists(VIDEO_PATH):
    video_name_stem_override = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
else:
    video_name_stem_override = "video_not_found"

# Create separate directories for NMS and raw detections
OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem_override}{crop_str_override}_{timestamp_run}")
OUTPUT_DIR_NMS = os.path.join(OUTPUT_DIR_RUN, "nms_filtered")
OUTPUT_DIR_RAW = os.path.join(OUTPUT_DIR_RUN, "raw_detections")

if SAVE_ANNOTATED_FRAMES:
    os.makedirs(OUTPUT_DIR_NMS, exist_ok=True)
    os.makedirs(OUTPUT_DIR_RAW, exist_ok=True)
    print(f"  NMS Output: {OUTPUT_DIR_NMS}")
    print(f"  Raw Output: {OUTPUT_DIR_RAW}")

print("Crop-based inference setup applied.")
print("-" * 50)

crop_detection_model = model
print("Crop detection model setup complete.")

# Finalize Class Names Map for Annotation
class_names_map_for_annotation = vp.class_id_to_name_map if hasattr(vp, 'class_id_to_name_map') else {}
default_class_label_for_annotation = 'unknown'

if not class_names_map_for_annotation:
    print(f"WARNING: Class name map from VideoProcessor (vp.class_id_to_name_map) is empty.")
    if hasattr(model, 'names') and model.names and isinstance(model.names, dict):
        class_names_map_for_annotation = model.names
        print(f"  Fell back to using model.names for annotation: {class_names_map_for_annotation}")
    else:
        class_names_map_for_annotation = {i: f"{default_class_label_for_annotation}_{i}" for i in range(80)}
        print(f"  CRITICAL FALLBACK: Using very generic names for annotation (e.g., '{default_class_label_for_annotation}_0')")

print(f"\nFinal class name map to be used for annotations: "
      f"{class_names_map_for_annotation if class_names_map_for_annotation else 'ERROR - MAP IS EMPTY'}")

print(f"\n--- Starting Benchmark Processing ---")

# Get Video Properties
total_video_frames = 0
original_video_fps = 0.0
_cap_cv_check = cv2.VideoCapture(VIDEO_PATH)
if _cap_cv_check.isOpened():
    total_video_frames = int(_cap_cv_check.get(cv2.CAP_PROP_FRAME_COUNT))
    original_video_fps = _cap_cv_check.get(cv2.CAP_PROP_FPS)
    _cap_cv_check.release()
else:
    print(f"Error: OpenCV could not open video {VIDEO_PATH} to get properties.")
    raise IOError(f"Failed to get video properties for {VIDEO_PATH}")

if total_video_frames == 0 or original_video_fps == 0:
    print("Warning: Could not reliably get video properties. Using defaults.")
    total_video_frames = 1000000
    original_video_fps = TARGET_FPS if TARGET_FPS > 0 else 25.0

print(f"Video Properties: ~{total_video_frames} frames, {original_video_fps:.2f} FPS (Original)")
frame_skip_interval = FRAME_INTERVAL_CONFIG

# Initialize Enhanced Counters
frames_actually_processed = 0
total_preprocessing_time_ms_std = 0
total_inference_time_ms = 0
total_enhancement_time_ms = 0
total_nms_combination_time_ms = 0
total_raw_processing_time_ms = 0
raw_frame_index_overall = 0
total_crops_generated = 0
total_raw_detections = 0
total_nms_detections = 0

# OpenCV VideoCapture for getting original frames for annotation
cap_for_originals = cv2.VideoCapture(VIDEO_PATH)
if not cap_for_originals.isOpened():
    print(f"CRITICAL ERROR: OpenCV could not open video {VIDEO_PATH} for original frames.")
    raise IOError(f"Failed to open video for original frames: {VIDEO_PATH}")

# Video Reading Loop Setup
reader_frame_source = None
segment_iterator = range(1)
pvr_segment_details = {}

print(f"PVR: Initializing for {VIDEO_PATH}, target: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")
reader_frame_source = PyVideoReader(VIDEO_PATH, device='cpu',
                                    resize_shorter_side=MODEL_INPUT_SIZE,
                                    resize_longer_side=MODEL_INPUT_SIZE)
pvr_segment_details['num_segments'] = math.ceil(total_video_frames / PYVIDEOREADER_DECODE_SEGMENT_SIZE)
segment_iterator = range(pvr_segment_details['num_segments'])
print(f"PVR Source: Segments={pvr_segment_details['num_segments']}")

def extract_raw_detections_from_crops(element_crops, conf_threshold=BASE_MODEL_CONF_FOR_CROPS):
    """Extract raw detections from CropElement objects and transform to original image coordinates"""
    raw_boxes = []
    raw_confidences = []
    raw_classes = []
    
    print(f"    DEBUG: Extracting from {len(element_crops.crops)} crops...")
    
    # Get scaling factors to map from resized image back to original
    original_image = element_crops.crops[0].source_image if element_crops.crops else None
    resized_image = element_crops.crops[0].source_image_resized if element_crops.crops else None
    
    scale_factor_x = 1.0
    scale_factor_y = 1.0
    
    if original_image is not None and resized_image is not None:
        scale_factor_x = original_image.shape[1] / resized_image.shape[1]  # width scaling
        scale_factor_y = original_image.shape[0] / resized_image.shape[0]  # height scaling
        print(f"    DEBUG: Scale factors - X: {scale_factor_x:.3f}, Y: {scale_factor_y:.3f}")
    
    crops_with_detections = 0
    total_detections_found = 0
    
    for crop_idx, crop in enumerate(element_crops.crops):
        # Check if crop has detections - fix the boolean array issue
        has_detections = (hasattr(crop, 'detected_xyxy') and 
                         crop.detected_xyxy is not None and 
                         len(crop.detected_xyxy) > 0)
        
        has_confidences = (hasattr(crop, 'detected_conf') and 
                          crop.detected_conf is not None and 
                          len(crop.detected_conf) > 0)
        
        has_classes = (hasattr(crop, 'detected_cls') and 
                      crop.detected_cls is not None and 
                      len(crop.detected_cls) > 0)
        
        if not (has_detections and has_confidences and has_classes):
            continue
        
        crops_with_detections += 1
        crop_detections = len(crop.detected_xyxy)
        total_detections_found += crop_detections
        
        print(f"    DEBUG: Crop {crop_idx} has {crop_detections} detections")
        
        # Process each detection in this crop
        for det_idx in range(crop_detections):
            try:
                conf = float(crop.detected_conf[det_idx])
                if conf < conf_threshold:
                    continue
                    
                # Get crop-relative coordinates
                x1_crop, y1_crop, x2_crop, y2_crop = crop.detected_xyxy[det_idx]
                cls_id = int(crop.detected_cls[det_idx])
                
                # Get crop position in resized image
                crop_x_start = crop.x_start
                crop_y_start = crop.y_start
                
                # Transform to resized image coordinates
                x1_resized = x1_crop + crop_x_start
                y1_resized = y1_crop + crop_y_start
                x2_resized = x2_crop + crop_x_start
                y2_resized = y2_crop + crop_y_start
                
                # Transform to original image coordinates
                x1_orig = x1_resized * scale_factor_x
                y1_orig = y1_resized * scale_factor_y
                x2_orig = x2_resized * scale_factor_x
                y2_orig = y2_resized * scale_factor_y
                
                raw_boxes.append([x1_orig, y1_orig, x2_orig, y2_orig])
                raw_confidences.append(conf)
                raw_classes.append(cls_id)
                
            except Exception as e:
                print(f"    DEBUG: Error processing detection {det_idx} in crop {crop_idx}: {e}")
                continue
    
    print(f"    DEBUG: {crops_with_detections}/{len(element_crops.crops)} crops had detections")
    print(f"    DEBUG: Total detections found: {total_detections_found}, above threshold: {len(raw_boxes)}")
    
    return raw_boxes, raw_confidences, raw_classes

def annotate_detections(frame, boxes, confidences, classes, class_map, color, prefix=""):
    """Annotate frame with detections"""
    annotated_frame = frame.copy()
    num_detections = 0
    
    for i in range(len(boxes)):
        conf = confidences[i]
        if conf < POSTPROCESS_CONF_THRESHOLD:
            continue
            
        num_detections += 1
        x1, y1, x2, y2 = map(int, boxes[i])
        cls_id = classes[i]
        
        # Ensure coordinates are within frame bounds
        frame_h, frame_w = frame.shape[:2]
        x1 = max(0, min(x1, frame_w-1))
        y1 = max(0, min(y1, frame_h-1))
        x2 = max(0, min(x2, frame_w-1))
        y2 = max(0, min(y2, frame_h-1))
        
        if x2 <= x1 or y2 <= y1:
            continue
        
        label_str = class_map.get(cls_id, f"{default_class_label_for_annotation}_{cls_id}")
        
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(annotated_frame, f"{prefix}{label_str}:{conf:.2f}", 
                   (x1, y1-10 if y1>20 else y1+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    return annotated_frame, num_detections

# Main Loop
for seg_idx in segment_iterator:
    if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
        break

    current_pvr_segment_frames_rgb = None
    pvr_current_segment_start_idx = 0

    pvr_current_segment_start_idx = seg_idx * PYVIDEOREADER_DECODE_SEGMENT_SIZE
    segment_end_raw_idx = min((seg_idx + 1) * PYVIDEOREADER_DECODE_SEGMENT_SIZE, total_video_frames)
    if pvr_current_segment_start_idx >= segment_end_raw_idx:
        continue
    
    print(f"\nPVR: Decoding segment {seg_idx+1}/{pvr_segment_details['num_segments']} "
            f"(raw frames {pvr_current_segment_start_idx} to {segment_end_raw_idx-1})")
    current_pvr_segment_frames_rgb = reader_frame_source.decode(start_frame=pvr_current_segment_start_idx,
                                                                end_frame=segment_end_raw_idx)
    if current_pvr_segment_frames_rgb is None or current_pvr_segment_frames_rgb.shape[0] == 0:
        print(f"PVR: Segment {seg_idx+1} returned no frames.")
        continue
    print(f"PVR: Decoded {current_pvr_segment_frames_rgb.shape[0]} frames from segment.")
    
    frame_idx_in_current_pvr_segment = -1

    while True:
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break
        
        current_original_video_frame_idx = -1
        is_input_rgb = False

        frame_idx_in_current_pvr_segment += 1
        if frame_idx_in_current_pvr_segment >= current_pvr_segment_frames_rgb.shape[0]:
            break
        current_original_video_frame_idx = pvr_current_segment_start_idx + frame_idx_in_current_pvr_segment
        is_input_rgb = True 

        raw_frame_index_overall = current_original_video_frame_idx

        if current_original_video_frame_idx % frame_skip_interval != 0:
            continue

        # Get original BGR frame
        original_frame_bgr = None
        cap_for_originals.set(cv2.CAP_PROP_POS_FRAMES, current_original_video_frame_idx)
        ret_orig, original_frame_bgr = cap_for_originals.read()
        if not ret_orig or original_frame_bgr is None:
            print(f"Warning: OpenCV could not read original frame at index {current_original_video_frame_idx} for drawing.")
            continue

        frames_actually_processed += 1
        print(f"Processing Original Frame #{current_original_video_frame_idx} (Actual Processed Frame #{frames_actually_processed})")

        image_to_infer_on_bgr = original_frame_bgr.copy()
        
        # Inference Block - Crop-based inference
        current_inf_block_start_time = time.monotonic()
        print(f"  Running crop-based prediction...")
        
        try:
            # Create crops and run inference
            element_crops = MakeCropsDetectThem(
                image=image_to_infer_on_bgr,
                model=crop_detection_model,
                imgsz=MODEL_INPUT_SIZE,
                conf=BASE_MODEL_CONF_FOR_CROPS,
                iou=POSTPROCESS_IOU_THRESHOLD,
                classes_list=None,
                segment=False,
                shape_x=CROP_WIDTH,
                shape_y=CROP_HEIGHT,
                overlap_x=OVERLAP_WIDTH_PERCENTAGE,
                overlap_y=OVERLAP_HEIGHT_PERCENTAGE,
                show_crops=False,
                show_processing_status=ENABLE_PROGRESS_TRACKING,
                resize_initial_size=True,
                memory_optimize=USE_MEMORY_OPTIMIZE,
                batch_inference=USE_BATCH_INFERENCE
            )
            
            num_crops = len(element_crops.crops)
            total_crops_generated += num_crops
            print(f"    Generated {num_crops} crops from frame")
            
            # Extract raw detections from CropElement objects
            raw_processing_start = time.monotonic()
            raw_boxes, raw_confidences, raw_classes = extract_raw_detections_from_crops(element_crops, BASE_MODEL_CONF_FOR_CROPS)
            raw_processing_time_ms = (time.monotonic() - raw_processing_start) * 1000
            total_raw_processing_time_ms += raw_processing_time_ms
            
            num_raw_detections = len(raw_boxes)
            total_raw_detections += num_raw_detections
            print(f"    Raw detections: {num_raw_detections}")
            
            # Combine detections using NMS
            nms_start_time = time.monotonic()
            result = CombineDetections(
                element_crops=element_crops,
                nms_threshold=NMS_THRESHOLD,
                match_metric=MATCH_METRIC,
                class_agnostic_nms=CLASS_AGNOSTIC_NMS,
                intelligent_sorter=INTELLIGENT_SORTER,
                sorter_bins=SORTER_BINS
            )
            nms_time_ms = (time.monotonic() - nms_start_time) * 1000
            total_nms_combination_time_ms += nms_time_ms
            
            # Count NMS filtered detections
            num_nms_detections = len(result.filtered_boxes) if hasattr(result, 'filtered_boxes') else 0
            total_nms_detections += num_nms_detections
            print(f"    NMS filtered detections: {num_nms_detections}")
            
        except Exception as e:
            print(f"  ERROR during crop-based inference: {e}")
            import traceback
            traceback.print_exc()
            continue
        
        current_inf_block_time_ms = (time.monotonic() - current_inf_block_start_time) * 1000
        total_inference_time_ms += current_inf_block_time_ms
        print(f"  Inference times - Total: {current_inf_block_time_ms:.2f}ms, Raw: {raw_processing_time_ms:.2f}ms, NMS: {nms_time_ms:.2f}ms")

        # Create annotated frames for both raw and NMS detections
        if SAVE_ANNOTATED_FRAMES:
            # Raw detections (red color)
            raw_annotated_frame, num_raw_drawn = annotate_detections(
                original_frame_bgr, raw_boxes, raw_confidences, raw_classes,
                class_names_map_for_annotation, (0, 0, 255), "R-"
            )
            
            # NMS filtered detections (green color)
            nms_boxes = result.filtered_boxes if hasattr(result, 'filtered_boxes') else []
            nms_confidences = result.filtered_confidences if hasattr(result, 'filtered_confidences') else []
            nms_classes = result.filtered_classes_id if hasattr(result, 'filtered_classes_id') else []
            
            nms_annotated_frame, num_nms_drawn = annotate_detections(
                original_frame_bgr, nms_boxes, nms_confidences, nms_classes,
                class_names_map_for_annotation, (0, 255, 0), "C-"
            )
            
            # Save both versions
            frame_filename = f"frame_{current_original_video_frame_idx:06d}.jpg"
            cv2.imwrite(os.path.join(OUTPUT_DIR_RAW, frame_filename), raw_annotated_frame)
            cv2.imwrite(os.path.join(OUTPUT_DIR_NMS, frame_filename), nms_annotated_frame)
            
            print(f"  Saved - Raw: {num_raw_drawn} detections, NMS: {num_nms_drawn} detections")

        if DISPLAY_INTERACTIVE_FRAMES:
            display_image_jupyter(nms_annotated_frame, title=f"NMS Filtered - Frame #{current_original_video_frame_idx}")
            if num_raw_detections > 0:
                display_image_jupyter(raw_annotated_frame, title=f"Raw Detections - Frame #{current_original_video_frame_idx}")
        
        # Memory cleanup
        del element_crops, result, raw_annotated_frame, nms_annotated_frame
        gc.collect()
        
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break

    if current_pvr_segment_frames_rgb is not None:
        del current_pvr_segment_frames_rgb
        gc.collect()

# Release Resources
if reader_frame_source: 
    del reader_frame_source
    gc.collect()
if cap_for_originals.isOpened(): 
    cap_for_originals.release()
gc.collect()

# Enhanced Final Summary
print("\n--- Enhanced Benchmark Summary ---")
if frames_actually_processed > 0:
    avg_inference_block_ms = total_inference_time_ms / frames_actually_processed
    avg_nms_ms = total_nms_combination_time_ms / frames_actually_processed
    avg_raw_processing_ms = total_raw_processing_time_ms / frames_actually_processed
    avg_crops_per_frame = total_crops_generated / frames_actually_processed
    avg_raw_detections_per_frame = total_raw_detections / frames_actually_processed
    avg_nms_detections_per_frame = total_nms_detections / frames_actually_processed
    
    print(f"**Processing Statistics:**")
    print(f"  Total Video Frames: {total_video_frames}")
    print(f"  Frames Processed: {frames_actually_processed}")
    print(f"  Total Crops Generated: {total_crops_generated} (avg: {avg_crops_per_frame:.1f}/frame)")
    print(f"  Total Raw Detections: {total_raw_detections} (avg: {avg_raw_detections_per_frame:.1f}/frame)")
    print(f"  Total NMS Detections: {total_nms_detections} (avg: {avg_nms_detections_per_frame:.1f}/frame)")
    if total_raw_detections > 0:
        print(f"  NMS Reduction Rate: {((total_raw_detections - total_nms_detections) / total_raw_detections) * 100:.1f}%")
    
    print(f"\n**Timing Performance:**")
    print(f"  Raw Processing: {avg_raw_processing_ms:.2f} ms/frame")
    print(f"  NMS Combination: {avg_nms_ms:.2f} ms/frame")
    print(f"  Total Inference Block: {avg_inference_block_ms:.2f} ms/frame")
    
    total_avg_ms_cycle = avg_inference_block_ms
    print(f"  Total Cycle Time: {total_avg_ms_cycle:.2f} ms/frame")
    
    if total_avg_ms_cycle > 0: 
        equivalent_fps = 1000 / total_avg_ms_cycle
        print(f"  Equivalent FPS: {equivalent_fps:.2f}")
        
        # Performance breakdown
        print(f"\n**Performance Breakdown:**")
        print(f"  Raw Processing: {(avg_raw_processing_ms/total_avg_ms_cycle)*100:.1f}%")
        print(f"  NMS Combination: {(avg_nms_ms/total_avg_ms_cycle)*100:.1f}%")
else:
    print("No frames were processed.")

if SAVE_ANNOTATED_FRAMES:
    raw_files = len(os.listdir(OUTPUT_DIR_RAW)) if os.path.exists(OUTPUT_DIR_RAW) else 0
    nms_files = len(os.listdir(OUTPUT_DIR_NMS)) if os.path.exists(OUTPUT_DIR_NMS) else 0
    print(f"\n**Output Files:**")
    print(f"  Raw detections: {raw_files} files in {OUTPUT_DIR_RAW}")
    print(f"  NMS filtered: {nms_files} files in {OUTPUT_DIR_NMS}")

print("\nEnhanced notebook execution finished.")


2025-09-06 11:16:24.489 | DEBUG    | models.model_loader:load_model:11 - Loading model from: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine onto device: cuda
2025-09-06 11:16:24.489 | DEBUG    | models.model_loader:load_model:47 - Detected .engine file, using TensorRT runtime via YOLO wrapper.
2025-09-06 11:16:24.490 | DEBUG    | models.model_loader:load_model:50 - TensorRT engine loaded successfully from: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine


  Crop Params: Crop HxW=640x640, Overlap=15%, Post-Conf=0.15
  Override: CONF_THRESHOLD_OVERRIDE (for base model on crops) set to 0.08
  NMS Output: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_20250906_1116\nms_filtered
  Raw Output: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_20250906_1116\raw_detections
Crop-based inference setup applied.
--------------------------------------------------
Crop detection model setup complete.
Loading C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine for TensorRT inference...
  Fell back to using model.names for annotation: {0: 'Bus', 1: 'Cyclist', 2: 'Light Vehicle', 3: 'Long Truck', 4: 'Medium Truck', 5: 'Minibus Taxi', 6: 'Motorcycle', 7: 'Pedestrian', 8: 'Short Truck'}

Final class name map to be used for annotation